# 02 — What we measure

---

Every feature here is a property of the **letter sequence itself**, independent of what the genome codes for.

| Feature | Definition |
|---|---|
| **GC content** | fraction of letters that are G or C |
| **Dinucleotide frequency** | how often a given pair of adjacent letters occurs |
| **O/E ratio** | observed pair frequency divided by what chance predicts |
| **GC3 / GC12** | GC content at the third codon position / at the first and second |

### The pairs tracked

| Pair | Written | Note |
|---|---|---|
| C then G | CpG | the motif of interest |
| G then C | GpC | |
| G then G | GpG | |
| C then C | CpC | |

### Why O/E rather than raw counts

A GC-rich genome contains more CpG by chance alone, so we divide by chance expectation:

$$ \mathrm{O/E}(\mathrm{CpG}) = \frac{f(\mathrm{CpG})}{f(\mathrm{C}) \times f(\mathrm{G})} $$

Near **1** = as common as chance predicts. **Below 1** = the virus is avoiding that pair.

### Why split GC by codon position

- **GC3** — third position; most changes are synonymous and leave the protein unchanged.
- **GC12** — first and second; most changes do alter the protein.

Comparing how fast the two move separates pressure on the RNA from pressure on the protein — the same logic as dS versus dN. The paper reports GC3 changing **5.65× faster** than GC12 in influenza.

---

In [ ]:
import sys, os, subprocess
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = os.path.abspath("..")          # repo root; the R scripts expect to run from here
pd.set_option("display.width", 130)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
PAL = {"A": "#2C7FB8", "B": "#C62824"}    # same colours as the R figures

def run_r(script="R/figures.R"):
    """Run the R figure script from the repo root and report what it wrote."""
    r = subprocess.run(["Rscript", script], capture_output=True, text=True, cwd=ROOT)
    print(r.stdout.strip() or r.stderr.strip()[-400:])

def show(fig):
    """Display a figure from results/figures/ inline."""
    display(Image(os.path.join(ROOT, "results", "figures", fig)))
from rsv import composition

## 2.1  The functions, on a short example

All measurements come from `src/rsv/composition.py`. Ambiguous positions are skipped rather than guessed.

In [ ]:
demo = "ATGGCGTTAACGCGTAAATTACGATAG"
f = composition.all_features(demo, coding=True)
for k in ["gc_content", "gc3", "gc12", "per1k_CG", "oe_CG"]:
    print(f"{k:12s} {f[k]:.4f}")

## 2.2  The measurements across all genomes

`GENOME` means the ten coding regions concatenated, which is how the paper defines genome-level GC, GC3 and GC12 — it keeps the reading frame well defined.

In [ ]:
g = pd.read_csv("../data/processed/rsv_gene_features.tsv", sep="\t")
genome = g[g.gene == "GENOME"]
cols = ["gc_content", "gc3", "gc12", "oe_CG", "oe_GC", "oe_GG", "oe_CC", "oe_TA"]
genome.groupby("Subtype")[cols].mean().round(4)

### Reading that table

- **CpG O/E sits far below 1 in both subtypes** (0.236 in A, 0.242 in B) — RSV carries only a quarter of the CpG pairs its base composition would predict. GpC, GpG and CpC sit near 1, so this is not a general shortage of G and C.
- **UpA is only mildly suppressed** (0.898 / 0.899) — and that is the informative contrast. UpA is the one other dinucleotide with experimental evidence of restricting RNA virus replication, through a recognition site separate from the CpG one (Goonawardane et al. 2021, mSphere). If RSV were under broad pressure against any immune-visible motif, UpA should be suppressed too. It is not, so the pressure here is CpG-specific — consistent with a ZAP-like pathway rather than general RNA degradation. GpC and ApU, by contrast, are the true controls: reversing the letters leaves base composition unchanged and abolishes the effect experimentally.
- **GC3 is much lower than GC12** — typical, because third positions are free to drift when changing them does not change the protein.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for s in "AB":
    x = genome[genome.Subtype == s]
    ax[0].hist(x.gc_content, bins=70, alpha=.6, color=PAL[s], label=f"RSV-{s}")
ax[0].set_xlabel("GC content"); ax[0].set_ylabel("genomes")
ax[0].set_title("GC content"); ax[0].legend(frameon=False)
oe = ["oe_CG", "oe_GC", "oe_GG", "oe_CC", "oe_TA"]
names = ["CpG", "GpC", "GpG", "CpC", "UpA"]
pos = np.arange(len(oe))
for i, s in enumerate("AB"):
    x = genome[genome.Subtype == s]
    ax[1].bar(pos + (i - .5) * .38, [x[c].mean() for c in oe], width=.38,
              color=PAL[s], label=f"RSV-{s}", edgecolor="0.3", linewidth=.3)
ax[1].axhline(1, ls="--", lw=1, c="k")
ax[1].set_xticks(pos); ax[1].set_xticklabels(names)
ax[1].set_ylabel("mean O/E"); ax[1].set_title("Dinucleotide O/E (dashed line = 1)")
ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

Only **CpG** falls clearly below the dashed line. That is the pattern reported across RNA viruses that circulate in humans.